In [1]:
!pip install pennylane pennylane-lightning


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 76.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 13.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 92.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 5.2 MB/s eta 0:00:00


In [15]:
# ==========================================================
# Hybrid Quantum-Classical Model:
# (A) PlantVillage - Crop Disease Detection (CNN + QCNN)
# (B) Soil Fertility Prediction (MLP + QCNN)
# ==========================================================

!pip install pennylane torch torchvision scikit-learn --quiet

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import pennylane as qml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ==========================================================
# 1. DATASET PATHS
# ==========================================================
plant_train_dir = "/kaggle/input/plantvillage/PlantVillage/train"
plant_val_dir   = "/kaggle/input/plantvillage/PlantVillage/val"
soil_path       = "/kaggle/input/soil-fertility-dataset/dataset1.csv"  # adjust if filename differs

# ==========================================================
# 2. PLANTVILLAGE DATA (CROP DISEASE)
# ==========================================================
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder(root=plant_train_dir, transform=transform)
val_dataset   = datasets.ImageFolder(root=plant_val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)

disease_classes = train_dataset.classes
print("Disease classes:", disease_classes)

# ==========================================================
# 3. SOIL DATASET (AUTO DETECT TARGET COLUMN)
# ==========================================================
soil_df = pd.read_csv(soil_path)
print("Soil dataset columns:", soil_df.columns)

# Assume last column = target
target_col = soil_df.columns[-1]
X = soil_df.drop(target_col, axis=1).values
y = soil_df[target_col].values

# Encode labels if categorical
if soil_df[target_col].dtype == "object":
    le = LabelEncoder()
    y = le.fit_transform(y)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_val   = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val   = torch.tensor(y_val, dtype=torch.long).to(device)

# ==========================================================
# 4. QUANTUM LAYER (QCNN)
# ==========================================================
# ==========================================================
# 4. QUANTUM LAYER (QCNN) - FIXED
# ==========================================================
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Encode classical inputs
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    # Variational ansatz
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    # Measurement
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# Define weight shapes for TorchLayer
weight_shapes = {"weights": (3, n_qubits)}

# Now wrap it correctly
qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

# ==========================================================
# 5. HYBRID MODELS
# ==========================================================

# (A) PlantVillage Disease Detection
class HybridPlantModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = models.resnet18(weights=None)
        self.cnn.fc = nn.Linear(self.cnn.fc.in_features, n_qubits)
        self.qnn = qlayer
        self.fc = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.qnn(x)
        x = self.fc(x)
        return x

# (B) Soil Fertility Prediction
class HybridSoilModel(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, n_qubits)
        self.qnn = qlayer
        self.fc2 = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.tanh(x)
        x = self.qnn(x)
        x = self.fc2(x)
        return x

# ==========================================================
# 6. TRAINING FUNCTIONS
# ==========================================================
def train_model(model, loader, criterion, optimizer, epochs=1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

def evaluate_model(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    print(classification_report(y_true, y_pred))
    print("Accuracy:", accuracy_score(y_true, y_pred))

# ==========================================================
# 7. RUN TRAINING
# ==========================================================

# --- PlantVillage ---
plant_model = HybridPlantModel(num_classes=len(disease_classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(plant_model.parameters(), lr=1e-4)

print("\nTraining PlantVillage Model...")
train_model(plant_model, train_loader, criterion, optimizer, epochs=2)
print("Evaluating PlantVillage Model...")
evaluate_model(plant_model, val_loader)

# --- Soil Dataset ---
num_classes_soil = len(np.unique(y))
soil_model = HybridSoilModel(input_dim=X.shape[1], num_classes=num_classes_soil).to(device)
criterion_soil = nn.CrossEntropyLoss()
optimizer_soil = optim.Adam(soil_model.parameters(), lr=1e-3)

print("\nTraining Soil Model...")
for epoch in range(5):
    optimizer_soil.zero_grad()
    outputs = soil_model(X_train)
    loss = criterion_soil(outputs, y_train)
    loss.backward()
    optimizer_soil.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print("Evaluating Soil Model...")
soil_model.eval()
with torch.no_grad():
    preds = torch.argmax(soil_model(X_val), dim=1)
print(classification_report(y_val.cpu(), preds.cpu()))
print("Accuracy:", accuracy_score(y_val.cpu(), preds.cpu()))


Using device: cuda
Disease classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spott

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00       126
           1       0.00      0.00      0.00       125
           2       0.00      0.00      0.00        55
           3       0.00      0.00      0.00       329
           4       0.00      0.00      0.00       300
           5       0.00      0.00      0.00       210
           6       0.00      0.00      0.00       170
           7       0.00      0.00      0.00       103
           8       0.04      0.10      0.06       239
           9       0.00      0.00      0.00       197
          10       0.00      0.00      0.00       233
          11       0.00      0.00      0.00       236
          12       0.00      0.00      0.00       276
          13       0.19      0.60      0.28       215
          14       0.00      0.00      0.00        84
          15       0.58      0.98      0.73      1102
          16       0.26      0.88      0.40       459
          17       0.00    

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [5]:
pip install reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.3 MB/s eta 0:00:0000:010:01
Note: you may need to restart the kernel to use updated packages.


In [19]:
# ===============================
# Hybrid Quantum-Classical CNN for Plant Disease + Soil Data
# ===============================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image

import pennylane as qml
from pennylane import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# -----------------
# CONFIG
# -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

batch_size = 16
num_epochs = 5   # change higher (20+) for better results
soil_epochs = 10
n_qubits = 4

# -----------------
# DATASET (PlantVillage)
# -----------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder(root="/kaggle/input/plantvillage/PlantVillage/train", transform=transform)
val_data   = datasets.ImageFolder(root="/kaggle/input/plantvillage/PlantVillage/val", transform=transform)

class_names = train_data.classes
print("Disease classes:", class_names)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size, shuffle=False)

# -----------------
# Quantum Circuit
# -----------------
dev = qml.device("default.qubit", wires=n_qubits)  # Simulator, change to ibmq/qiskit for real QC

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.rand((1, n_qubits)))

    def forward(self, x):
        out = []
        for elem in x:
            # ensure inputs are correct size
            elem = elem[:n_qubits]
            result = quantum_circuit(elem, self.weights)
            out.append(torch.tensor(result, dtype=torch.float32, device=device))
        return torch.stack(out)

# -----------------
# Hybrid QCNN Model
# -----------------
class HybridQCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 30 * 30, n_qubits),
        )
        self.q_layer = QuantumLayer()
        self.fc = nn.Linear(n_qubits, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.q_layer(x)
        x = self.fc(x)
        return x

plant_model = HybridQCNN(len(class_names)).to(device)

# -----------------
# TRAINING FUNCTION (PlantVillage)
# -----------------
def train_plant_model(model, train_loader, val_loader, num_epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print("Classification Report (PlantVillage):")
    print(classification_report(all_labels, all_preds))
    print("Accuracy:", accuracy_score(all_labels, all_preds))

# -----------------
# Soil Dataset
# -----------------
soil_data = pd.read_csv("/kaggle/input/soil-fertility-dataset/dataset1.csv")
print("Soil dataset columns:", soil_data.columns)

X = soil_data.drop("Output", axis=1).values
y = soil_data["Output"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class SoilModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, n_qubits)
        self.q_layer = QuantumLayer()
        self.fc2 = nn.Linear(n_qubits, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.q_layer(x)
        x = self.fc2(x)
        return x

soil_model = SoilModel(X_train.shape[1], len(np.unique(y))).to(device)

# -----------------
# TRAINING FUNCTION (Soil)
# -----------------
def train_soil_model(model, X_train, y_train, X_test, y_test, num_epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X_test), dim=1)
        print("Classification Report (Soil):")
        print(classification_report(y_test.cpu(), preds.cpu()))
        print("Accuracy:", accuracy_score(y_test.cpu(), preds.cpu()))

# -----------------
# PREDICT FROM SINGLE IMAGE
# -----------------
def predict_image(model, image_path, class_names, device):
    img = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
    ])
    img = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(img)
        pred = output.argmax(dim=1).item()
    return class_names[pred]

# -----------------
# RUN TRAINING
# -----------------
print("\nTraining PlantVillage Model...")
train_plant_model(plant_model, train_loader, val_loader, num_epochs=num_epochs)

print("\nTraining Soil Model...")
train_soil_model(soil_model, X_train, y_train, X_test, y_test, num_epochs=soil_epochs)

# -----------------
# TEST SINGLE IMAGE
# -----------------
test_img_path = "/kaggle/input/plantvillage/PlantVillage/val/Tomato___healthy/0a1dcfb2-3c56-419f-9c66-2c9e7a49c2e0___RS_HL 1825.JPG"  # example
predicted = predict_image(plant_model, test_img_path, class_names, device)
print("Predicted Disease:", predicted)


Using device: cuda
Disease classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spott

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1, Loss: 1.1194
Epoch 2, Loss: 1.1188
Epoch 3, Loss: 1.1182
Epoch 4, Loss: 1.1176
Epoch 5, Loss: 1.1171
Epoch 6, Loss: 1.1165
Epoch 7, Loss: 1.1159
Epoch 8, Loss: 1.1153
Epoch 9, Loss: 1.1147
Epoch 10, Loss: 1.1141
Classification Report (Soil):
              precision    recall  f1-score   support

           0       0.44      1.00      0.61        78
           1       0.00      0.00      0.00        88
           2       0.00      0.00      0.00        10

    accuracy                           0.44       176
   macro avg       0.15      0.33      0.20       176
weighted avg       0.20      0.44      0.27       176

Accuracy: 0.4431818181818182


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/plantvillage/PlantVillage/val/Tomato___healthy/0a1dcfb2-3c56-419f-9c66-2c9e7a49c2e0___RS_HL 1825.JPG'

In [2]:
# ===============================
# Hybrid Quantum-Classical CNN for Plant Disease + Soil Data (Fixed)
# ===============================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
import pennylane as qml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# -----------------
# CONFIG
# -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

batch_size = 16
num_epochs = 20      # increase for better results
soil_epochs = 50
n_qubits = 6         # number of qubits
n_q_layers = 2       # variational layers in quantum circuit

# -----------------
# DATASET (PlantVillage)
# -----------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
])

train_data = datasets.ImageFolder(root="/kaggle/input/plantvillage/PlantVillage/train", transform=transform)
val_data   = datasets.ImageFolder(root="/kaggle/input/plantvillage/PlantVillage/val", transform=transform)

class_names = train_data.classes
print("Disease classes:", class_names)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=batch_size, shuffle=False)

# -----------------
# QUANTUM CIRCUIT
# -----------------
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    # Encode classical features into qubits
    for i in range(n_qubits):
        qml.RX(inputs[i], wires=i)
    # Variational layers with entanglement
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(0.01 * torch.randn(n_q_layers, n_qubits))

    def forward(self, x):
        out = []
        for elem in x:
            # Ensure input has correct size
            elem = elem[:n_qubits]
            res = quantum_circuit(elem, self.weights)
            out.append(torch.tensor(res, dtype=torch.float32, device=device))
        return torch.stack(out)

# -----------------
# HYBRID QCNN MODEL
# -----------------
class HybridQCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, 1),
            nn.ReLU(),
            nn.Flatten(),
            nn.LazyLinear(n_qubits)  # Automatically calculates flattened input
        )
        self.q_layer = QuantumLayer()
        self.fc = nn.Linear(n_qubits*2, num_classes)  # hybrid skip

    def forward(self, x):
        x_classical = torch.tanh(self.cnn(x)) * np.pi
        x_quantum = self.q_layer(x_classical)
        x = torch.cat([x_classical, x_quantum], dim=1)
        return self.fc(x)

plant_model = HybridQCNN(len(class_names)).to(device)

# -----------------
# TRAINING FUNCTION (PlantVillage)
# -----------------
def train_plant_model(model, train_loader, val_loader, num_epochs=10):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

    # Evaluate
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print("Classification Report (PlantVillage):")
    print(classification_report(all_labels, all_preds))
    print("Accuracy:", accuracy_score(all_labels, all_preds))

# -----------------
# SOIL DATASET
# -----------------
soil_data = pd.read_csv("/kaggle/input/soil-fertility-dataset/dataset1.csv")
X = soil_data.drop("Output", axis=1).values
y = soil_data["Output"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

class SoilModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, n_qubits)
        self.q_layer = QuantumLayer()
        self.fc2 = nn.Linear(n_qubits*2, output_dim)

    def forward(self, x):
        x_classical = torch.tanh(self.fc1(x)) * np.pi
        x_quantum = self.q_layer(x_classical)
        x = torch.cat([x_classical, x_quantum], dim=1)
        return self.fc2(x)

soil_model = SoilModel(X_train.shape[1], len(np.unique(y))).to(device)

# -----------------
# TRAINING FUNCTION (Soil)
# -----------------
def train_soil_model(model, X_train, y_train, X_test, y_test, num_epochs=20):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X_test), dim=1)
        print("Classification Report (Soil):")
        print(classification_report(y_test.cpu(), preds.cpu()))
        print("Accuracy:", accuracy_score(y_test.cpu(), preds.cpu()))

# -----------------
# PREDICT SINGLE IMAGE
# -----------------
def predict_image(model, image_path, class_names, device):
    img = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((128,128)),
        transforms.ToTensor(),
    ])
    img = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        output = model(img)
        pred = output.argmax(dim=1).item()
    return class_names[pred]

# -----------------
# RUN TRAINING
# -----------------
print("\nTraining PlantVillage Model...")
train_plant_model(plant_model, train_loader, val_loader, num_epochs=num_epochs)

print("\nTraining Soil Model...")
train_soil_model(soil_model, X_train, y_train, X_test, y_test, num_epochs=soil_epochs)

# -----------------
# TEST SINGLE IMAGE
# -----------------
test_img_path = "/kaggle/input/plantvillage/PlantVillage/val/Apple___Cedar_apple_rust/025b2b9a-0ec4-4132-96ac-7f2832d0db4a___FREC_C.Rust 3655.JPG"
predicted = predict_image(plant_model, test_img_path, class_names, device)
print("Predicted Disease:", predicted)


Using device: cuda
Disease classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spott

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [3]:
# ==============================
# IMPROVED SOIL MODEL (HYBRID QCNN)
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# -----------------
# CONFIG
# -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

n_qubits = 6        # number of qubits
n_q_layers = 3      # variational layers
soil_epochs = 100   # increase epochs for better accuracy
batch_size = 16

# -----------------
# LOAD SOIL DATA
# -----------------
soil_data = pd.read_csv("/kaggle/input/soil-fertility-dataset/dataset1.csv")
X = soil_data.drop("Output", axis=1).values
y = soil_data["Output"].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_test  = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test  = torch.tensor(y_test, dtype=torch.long).to(device)

# -----------------
# QUANTUM LAYER
# -----------------
import pennylane as qml

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RX(inputs[i], wires=i)
    qml.templates.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(0.01 * torch.randn(n_q_layers, n_qubits))

    def forward(self, x):
        out = []
        for elem in x:
            elem = elem[:n_qubits]  # truncate if needed
            res = quantum_circuit(elem, self.weights)
            out.append(torch.tensor(res, dtype=torch.float32, device=device))
        return torch.stack(out)

# -----------------
# IMPROVED SOIL MODEL
# -----------------
class ImprovedSoilModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu = nn.ReLU()
        self.q_layer = QuantumLayer()
        self.fc2 = nn.Linear(32 + n_qubits, 32)
        self.fc3 = nn.Linear(32, output_dim)

    def forward(self, x):
        x_classical = self.relu(self.bn1(self.fc1(x)))
        x_classical_scaled = torch.tanh(x_classical) * np.pi
        x_quantum = self.q_layer(x_classical_scaled)
        x = torch.cat([x_classical, x_quantum], dim=1)
        x = self.relu(self.fc2(x))
        return self.fc3(x)

soil_model = ImprovedSoilModel(X_train.shape[1], len(np.unique(y))).to(device)

# -----------------
# TRAINING FUNCTION
# -----------------
def train_soil_model(model, X_train, y_train, X_test, y_test, num_epochs=100):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        preds = torch.argmax(model(X_test), dim=1)
        print("\nClassification Report (Soil):")
        print(classification_report(y_test.cpu(), preds.cpu()))
        print("Accuracy:", accuracy_score(y_test.cpu(), preds.cpu()))

# -----------------
# RUN TRAINING
# -----------------
train_soil_model(soil_model, X_train, y_train, X_test, y_test, num_epochs=soil_epochs)


Using device: cuda
Epoch 10/100, Loss: 1.0821
Epoch 20/100, Loss: 0.9873
Epoch 30/100, Loss: 0.8827
Epoch 40/100, Loss: 0.7690
Epoch 50/100, Loss: 0.6580
Epoch 60/100, Loss: 0.5596
Epoch 70/100, Loss: 0.4812
Epoch 80/100, Loss: 0.4226
Epoch 90/100, Loss: 0.3819
Epoch 100/100, Loss: 0.3520

Classification Report (Soil):
              precision    recall  f1-score   support

           0       0.89      0.92      0.91        78
           1       0.83      0.90      0.86        88
           2       0.00      0.00      0.00        10

    accuracy                           0.86       176
   macro avg       0.57      0.61      0.59       176
weighted avg       0.81      0.86      0.83       176

Accuracy: 0.8579545454545454


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
